##Anemia Prediction
In this project we have anemic and non-anemic images. We want to build a model on these images and observe how well the model predict the conjuctival images if they are anaemic or not. Our evaluation metrics will be imported from the sklearn.metrics library ('classification_report').

We will use a deep learning model on the dataset. First we will import the two separate images and join them together. We will apply the deep learning model on it. We will tune hyperparameters such as the learning rate and the optimizer to improve the accuracy of the model.

We will use pretrained models on imagenet images and transfer their weights to train these images

In [ ]:
# Importing the necessary libraries
# import os
# from PIL import Image
# import numpy as np
# import pandas as pd
# import sklearn.metrics as metrics
# from sklearn.model_selection import train_test_split
# from keras.models import Sequential
# from keras.layers import Dense, Dropout, Flatten
# from keras.layers import Conv2D, MaxPooling2D
# from keras.optimizers import SGD
# from skimage.transform import resize
# from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, f1_score, precision_score, recall_score, roc_auc_score


from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report
from sklearn.metrics import f1_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import roc_auc_score
from sklearn.metrics import roc_curve
from sklearn.metrics import auc
from sklearn.utils import class_weight
import numpy as np
import os
from PIL import Image


In [ ]:
# Import the images from the two folders
anemic_folder = '//content/drive/MyDrive/Miscellaneous/Conjuctiva_images/Anemic'
non_anemic_folder = '/content/drive/MyDrive/Miscellaneous/Conjuctiva_images/Non-anemic'

anemic_images = []
anemic_labels = []

non_anemic_images = []
non_anemic_labels = []

# A loop to append label (1) as anemic
for filename in os.listdir(anemic_folder):
    im1 = Image.open(os.path.join(anemic_folder, filename)).convert('RGB').resize((256, 256))
    # Normalize the image
    im1 = np.array(im1)
    im1 = im1 / 255

    anemic_images.append(np.array(im1))
    anemic_labels.append(1)

for filename in os.listdir(non_anemic_folder):
    im2 = Image.open(os.path.join(non_anemic_folder, filename)).convert('RGB').resize((256, 256))
    # Normalize the image
    im2 = np.array(im2)
    im2 = im2 / 255
    non_anemic_images.append(np.array(im2))
    non_anemic_labels.append(0)


anemic_images=np.array(anemic_images)
non_anemic_images=np.array(non_anemic_images)

In [ ]:
X = np.concatenate((anemic_images, non_anemic_images))
y = np.concatenate((anemic_labels, non_anemic_labels))

In [ ]:
# Split the data into training and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Define the ensemble model
# from keras.models import Model
# from keras.layers import Input, Dense, Flatten, GlobalAveragePooling2D, Concatenate
# from keras.optimizers import Adam
from keras.applications import VGG16, ResNet50, InceptionV3, DenseNet201

In [ ]:
# Load the base models
vgg16 = VGG16(weights='imagenet', include_top=False, input_shape=(256, 256, 3))
resnet50 = ResNet50(weights='imagenet', include_top=False, input_shape=(256, 256, 3))
inceptionv3 = InceptionV3(weights='imagenet', include_top=False, input_shape=(256, 256, 3))
densenet201 = DenseNet201(weights='imagenet', include_top=False, input_shape=(256, 256, 3))

# Create the inputs of the ensemble model

inputs = Input(shape=(256, 256, 3))

# Extract features from the base models
vgg16_features = vgg16(inputs)
resnet50_features = resnet50(inputs)
inceptionv3_features = inceptionv3(inputs)
densenet201_features = densenet201(inputs)

# Flattening the features
vgg16_features = Flatten()(vgg16_features)
resnet50_features = Flatten()(resnet50_features)
inceptionv3_features = Flatten()(inceptionv3_features)
densenet201_features = Flatten()(densenet201_features)

# Concatenate the features
concatenated_features = Concatenate()([vgg16_features, resnet50_features, inceptionv3_features, densenet201_features])

# Add a dense layer for classification
outputs = Dense(1, activation='sigmoid')(concatenated_features)

# Create the ensemble model
ensemble_model = Model(inputs=inputs, outputs=outputs)

# Compile the model
ensemble_model.compile(optimizer=Adam(lr=1e-4), loss='binary_crossentropy', metrics=['accuracy'])




In [ ]:
#  Fit the model on the training data
# class_weights = class_weight.compute_class_weight(class_weight="balanced",classes=np.unique(y_train),y=y_train)
# class_weights = dict(enumerate(class_weights))
# ensemble_model.fit(X_train, y_train, epochs=10, batch_size=32, validation_data=(X_test, y_test), class_weight=class_weights)


class_weights = class_weight.compute_class_weight(class_weight="balanced",classes=np.unique(y_train),y=y_train)
classes = np.unique(y_train)
class_weights = dict(zip(classes, class_weights))
ensemble_model.fit(X_train, y_train, epochs=10, batch_size=32, validation_data=(X_test, y_test), class_weight=class_weights)



In [ ]:
# Make predictions on the test set
y_pred = ensemble_model.predict(X_test)

# Convert the predictions to binary labels
y_pred = np.round(y_pred).flatten()

# Evaluation Metrics
# Accuracy
acc = accuracy_score(y_test, y_pred)
print("Accuracy: {:.2f}%".format(acc*100))

# Confusion Matrix
confusion_matrix = confusion_matrix(y_test, y_pred)
print("Confusion Matrix: \n", confusion_matrix)

Classification Report
classification_report = classification_report(y_test, y_pred)
print("Classification Report: \n", classification_report)

F1 Score
f1_score = f1_score(y_test, y_pred)
print("F1 Score: {:.2f}".format(f1_score))

Precision
precision = precision_score(y_test, y_pred)
print("Precision: {:.2f}".format(precision))

Recall
recall = recall_score(y_test, y_pred)
print("Recall: {:.2f}".format(recall))

ROC AUC Score
roc_auc = roc_auc_score(y_test, y_pred)
print("ROC AUC: {:.2f}".format(roc_auc))

ROC Curve
fpr, tpr, thresholds = roc_curve(y_test, y_pred)
plt.plot(fpr, tpr, label='ROC curve (area = {:.2f})'.format(roc_auc))
plt.plot([0, 1], [0, 1], 'k--') # random predictions curve
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.0])
plt.xlabel('False Positive Rate or (1 - Specifity)')
plt.ylabel('True Positive Rate or (Sensitivity)')
plt.title('Receiver Operating Characteristic')
plt.legend(loc="lower right")

Save the model
model.save('anemia_detection')

Plot the ROC curve
plt.show()

Close the model
model.close()
